# Sesión 4 — Regresión lineal: qué hay dentro del `.fit()`

**Introducción al Análisis Avanzado de Datos con Python** · Univalle · Módulo 2 — Modelos supervisados

## Objetivos de hoy
Al terminar la sesión usted podrá:
1. Ajustar una regresión lineal, **leer sus coeficientes** y explicar una predicción como una suma.
2. Explicar qué minimiza el `.fit()` (la suma de residuales al cuadrado) y cómo lo encuentra: **fórmula cerrada** o **descenso de gradiente** con tasa de aprendizaje.
3. Evaluar un modelo de regresión con **MAE, RMSE y R²** contra una línea base, y **revisar sus supuestos** con los residuos (y la multicolinealidad con el VIF).
4. Usar **Ridge y Lasso** para controlar coeficientes y elegir variables, y reconocer cuándo un R² es demasiado bueno para ser cierto.

> 💾 **Antes de escribir nada:** menú *Archivo → Guardar una copia en Drive* y trabaje en esa copia; si no, lo que escriba se pierde al cerrar la pestaña. Si Colab avisa que el notebook no es de Google, elija *Ejecutar de todos modos*.

## Dónde estamos en el pipeline

```
 ┌────────┐   ┌──────────────────┐   ┌────────┐   ┌────────────┐   ┌────────────────┐
 │ DATOS  │ → │ PREPROCESAMIENTO │ → │ MODELO │ → │ EVALUACIÓN │ → │ INTERPRETACIÓN │
 └────────┘   └──────────────────┘   └────────┘   └────────────┘   └────────────────┘
                                       ▲▲▲▲▲▲▲▲
   HOY: un modelo nuevo y, por primera vez, abrimos su .fit() para ver la optimización por dentro.
```

## Cómo usar este notebook
- 🔍 **Predice** antes de ejecutar · 👀 **Mire** (piénselo un momento antes de seguir) · ✏️ **Completa**: desde hoy el andamiaje es **medio** (usted escribe bloques cortos con la firma dada) · 💥 **Trampa** · 🖼️ **Fichas** · ⚡ **Mini-reto**.

> **Dos datasets hoy.** *Insurance* (1 338 asegurados de EE. UU., cargo médico anual; el clásico de Lantz y de PyCaret) para ver el método con claridad. Luego **Saber 11, Valle del Cauca, 2022** (78 720 estudiantes; ICFES vía datos.gov.co) para aplicarlo con datos colombianos de verdad.


## 0. Setup

Baja del repositorio del curso los dos datasets y `figuras.py`. Si ya están en el disco de Colab, no los vuelve a bajar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
REPO = "https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/"
ARCHIVOS = {"insurance.csv": "datos/insurance.csv",
            "saber11_valle_2022.csv.gz": "datos/saber11_valle_2022.csv.gz",
            "figuras.py": "tools/figuras.py"}

def asegurar_datos(archivo, url_repo):
    """Deja un archivo en el disco de Colab: lo baja del repositorio del curso si aún no está."""
    from pathlib import Path
    import requests
    if Path(archivo).exists():
        return True
    try:
        r = requests.get(url_repo, timeout=120)
        if r.ok:
            Path(archivo).write_bytes(r.content)
            return True
    except Exception:
        pass
    print(f"⚠️ No pude bajar {archivo}. Use el Plan B (celda siguiente).")
    return False

for archivo, ruta in ARCHIVOS.items():
    asegurar_datos(archivo, REPO + ruta)
try:
    import figuras as fg
except ModuleNotFoundError:   # Plan B: GitHub no respondió y no llegó figuras.py
    from google.colab import files
    print("⚠️ No llegó figuras.py: súbalo (el profesor lo comparte por el canal del curso).")
    files.upload()
    import figuras as fg
print("Listo:", ", ".join(ARCHIVOS))

<details><summary><b>Plan B</b> — si la celda anterior avisó que no pudo bajar algún archivo</summary>

La celda siguiente revisa sola qué falta y solo en ese caso le pide subirlo (el instructor comparte los archivos por el grupo del curso).
</details>

In [ ]:
# Plan B: solo actúa si falta algún archivo. Con "Ejecutar todo" no se detiene.
from pathlib import Path
faltan = [a for a in ARCHIVOS if not Path(a).exists()]
if not faltan:
    print("✓ Todo disponible: no hace falta el Plan B.")
else:
    from google.colab import files
    print("⚠️ Faltan:", faltan, "→ súbalos")
    files.upload()

In [ ]:
fg.pipeline("modelo", modulo=2, subtitulo="HOY: un modelo nuevo (regresión lineal) y qué pasa dentro de su .fit()");

## 1. ¿Cuánto paga en salud un asegurado? — Insurance

| Columna | Qué es |
|---|---|
| `age`, `bmi`, `children` | edad, índice de masa corporal, número de hijos |
| `sex`, `smoker`, `region` | sexo, si fuma, región de EE. UU. |
| `charges` | **la y**: cargo médico anual en USD |

In [ ]:
ins = pd.read_csv("insurance.csv")
print(ins.shape, "| filas duplicadas:", ins.duplicated().sum())
ins = ins.drop_duplicates()          # la Sesión 2 en una línea: una fila repetida
ins.head()

👀 **Mire:** ¿cuánto paga **en promedio** al año alguien que fuma, comparado con alguien que no fuma? ¿El doble? ¿El triple? ¿Más?

In [ ]:
ins.groupby("smoker")["charges"].agg(["mean", "median", "count"]).round(0)

### 📘 Para entender — ¿qué es una regresión lineal y para qué sirve?

**Qué es.** Un modelo que predice un **número** (la *y*) como una **suma ponderada** de las variables de entrada:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_p x_p$$

- $\hat{y}$ ("y gorro") es la predicción; $x_1, \dots, x_p$ son las variables (edad, IMC, si fuma…).
- $\beta_0$ es el **intercepto**: lo que predice el modelo cuando todas las $x$ valen 0.
- Cada $\beta_j$ es un **coeficiente**: cuánto cambia la predicción cuando $x_j$ sube en una unidad y todo lo demás se queda igual.
- *Lineal* quiere decir que cada variable **suma** su aporte, siempre con el mismo peso; no que los datos tengan que caer en una recta perfecta.

**Para qué sirve.** Para dos cosas que en la práctica van juntas:

1. **Predecir** un valor numérico: cuánto, cuántos, cuánto tiempo.
2. **Explicar**: cada coeficiente se lee en las unidades de la *y* ("fumar suma unos 23 000 USD al año").

**Dónde se usa (ejemplos reales):**

| Pregunta | *y* | Algunas *x* |
|---|---|---|
| ¿Cuántas cajas de gaseosa venderá una tienda la próxima semana? | unidades vendidas | precio, temperatura, festivos, promociones |
| ¿Cuánto cuesta un apartamento en Cali? | precio | m², estrato, barrio, antigüedad, parqueadero |
| ¿Cuánta energía consumirá un hogar? | kWh al mes | estrato, número de personas, electrodomésticos |
| ¿Cuánto tardará un pedido en llegar? | horas | distancia, hora del día, día de la semana |
| ¿Cuánto saca un estudiante en el Saber 11? | puntaje global | colegio, jornada, educación de los padres (sección 5) |

**Cuándo es buena idea:** la *y* es numérica; necesita **explicar** el modelo a alguien (gerencia, un comité, un paper); tiene pocos datos; quiere una **línea base** sólida antes de probar modelos más complejos.

**Cuándo no basta:**

- La *y* es una categoría (sí/no, tipo de cliente) → clasificación (Sesión 5).
- El efecto de una variable **depende de otra** (fumar pesa más si además hay obesidad) o la relación es muy curva → árboles y bosques (Sesión 6), o agregar esas interacciones a mano.
- Hay valores atípicos enormes: como se elevan al cuadrado, unos pocos puntos pueden "jalar" la recta.

> ⚠️ **Predice no es causa.** Un coeficiente dice "con todo lo demás igual, los que tienen más de $x$ tienen más de $y$". Que $x$ *cause* $y$ es otra pregunta, que necesita otro diseño (un experimento, por ejemplo).

## 2. Antes del código: ¿qué recta es "la mejor"?

Empecemos con **una sola variable**: la edad, solo para quienes no fuman. Una regresión lineal propone

$$\hat{y} = \beta_0 + \beta_1 \cdot \text{edad}$$

¿Cuál de todas las rectas posibles? La que comete **menos error**. Cada punto tiene un **residual** $y_i - \hat{y}_i$ (la línea roja vertical) y la regresión minimiza la suma de sus cuadrados.

In [ ]:
no_fuman = ins[ins["smoker"] == "no"]
muestra = no_fuman.sample(40, random_state=3)
fg.recta_residuales(muestra["age"], muestra["charges"] / 1000, malas=((no_fuman["charges"].mean() / 1000, 0.0), None),
                    xlabel="edad", ylabel="cargo anual (miles de USD)");

### 📐 Fundamento

$$\text{pérdida}(\beta_0, \beta_1) = \frac{1}{n}\sum_{i=1}^{n}\big(y_i - (\beta_0 + \beta_1 x_i)\big)^2 \qquad \text{(error cuadrático medio, MSE)}$$

- $y_i$: el cargo real de la persona $i$ · $\beta_0 + \beta_1 x_i$: lo que la recta predice para ella.
- ¿Por qué **al cuadrado** y no el valor absoluto? Castiga más los errores grandes y, sobre todo, deja una pérdida con forma de **tazón liso**: tiene un solo fondo y se puede encontrar con álgebra o rodando cuesta abajo.

<details><summary><b>Para quien quiera más:</b> la fórmula cerrada</summary>

Con varias variables, en forma matricial, el fondo del tazón se despeja en un paso:
$$\hat{\beta} = (X^\top X)^{-1} X^\top y$$
Es lo que hace `LinearRegression` por dentro. Necesita invertir una matriz de $p \times p$ (con $p$ variables): con millones de filas o miles de variables se vuelve costoso, y ahí entra el descenso de gradiente.
</details>

![Dentro del .fit() — regresión lineal](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S04/img/dentro_del_fit_lineal.png)

### 📘 Cómo leer la ficha «Dentro del `.fit()`»

Cada modelo del curso se resume en los mismos cuatro renglones. Son las cuatro preguntas que hay que hacerle a **cualquier** algoritmo de ML:

| Renglón | Qué pregunta | En la regresión lineal | En el árbol (Sesiones 1–3) |
|---|---|---|---|
| **Qué minimiza** | ¿Qué cuenta como "equivocarse"? Esa cuenta se llama **función de pérdida**. | la suma de residuales al cuadrado | la impureza de Gini de cada corte |
| **Moviendo qué** | ¿Qué números ajusta el `.fit()`? Se llaman **parámetros** y los aprende de los datos. | los coeficientes $\beta_0, \beta_1, \dots$ | las preguntas y los cortes |
| **Cómo lo busca** | ¿Qué método usa para encontrar los mejores parámetros? Es el **optimizador**. | fórmula cerrada (o descenso de gradiente) | probar todos los cortes, uno a la vez |
| **Qué perilla controla la búsqueda** | ¿Qué decide usted *antes* de entrenar? Son los **hiperparámetros**. | $\eta$ (si hay gradiente) · $\alpha$ (en Ridge/Lasso) | `max_depth` |

Dos palabras que se confunden mucho:

- **Parámetro**: lo aprende el modelo (los $\beta$). Usted no lo escribe.
- **Hiperparámetro**: lo elige usted antes del `.fit()` (`alpha`, `max_depth`, la tasa de aprendizaje). Hay dos familias: los **del modelo** cambian *cuál* es la mejor respuesta (`alpha`); los **del optimizador** solo cambian *cómo* se llega a ella ($\eta$, número de pasos).

## 3. Cómo se encuentra el fondo: descenso de gradiente

Cada par $(\beta_0, \beta_1)$ es una recta posible, y cada recta tiene su error. Si dibujamos el error de **todas** las rectas, sale un paisaje con forma de tazón. Entrenar es encontrar el fondo.

(Para que el tazón se vea redondo usamos la edad **escalada** —restar la media, dividir por la desviación—, como en la Sesión 2, y el cargo en miles de USD.)

> 🧪 **Laboratorio** (para jugar después de clase, o ya si va adelantado): [**Laboratorio 3 · Descenso de gradiente**](https://nitobest.github.io/curso-aad-python/laboratorio/03_descenso.html) · la bolita rueda por el tazón del error; mueva la tasa de aprendizaje y vea cuándo se sale. Tres pestañas: **Mira**, **Juega** y **Reto** (prediga antes de correr). Funciona en el celular.

In [ ]:
x = ((no_fuman["age"] - no_fuman["age"].mean()) / no_fuman["age"].std()).values   # edad escalada
y_miles = (no_fuman["charges"] / 1000).values                                          # cargo en miles
fg.superficie_perdida(x, y_miles);

### La idea: rodar cuesta abajo
Desde cualquier punto del tazón, el **gradiente** $\nabla L$ dice hacia dónde sube la pérdida más rápido. Damos un pasito en la dirección contraria, y repetimos:

$$\beta \leftarrow \beta - \eta \, \nabla L(\beta)$$

Léala así: *"muévase un poquito ($\eta$) en contra de la pendiente"*. Para nuestra recta, las dos derivadas son:

$$\frac{\partial L}{\partial \beta_0} = \frac{2}{n}\sum(\hat{y}_i - y_i) \qquad \frac{\partial L}{\partial \beta_1} = \frac{2}{n}\sum(\hat{y}_i - y_i)\,x_i$$

### ✏️ Completa — escriba usted un paso del descenso
`residuo` es $\hat{y} - y$. Complete las dos líneas del `return` con las fórmulas de arriba.

In [ ]:
def paso_gradiente(b0, b1, x, y, eta):
    """Un paso de descenso de gradiente para la recta y ≈ b0 + b1·x (pérdida: MSE)."""
    residuo = (b0 + b1 * x) - y
    grad_b0 = ...   # TODO: 2 × promedio del residuo
    grad_b1 = ...   # TODO: 2 × promedio de (residuo × x)
    return b0 - eta * grad_b0, b1 - eta * grad_b1

b0, b1 = 0.0, 0.0                       # arrancamos en una recta plana en cero
for k in range(100):
    b0, b1 = paso_gradiente(b0, b1, x, y_miles, eta=0.1)
    if k in (0, 1, 4, 19, 99):
        print(f"paso {k+1:>3}: b0 = {b0:6.3f}   b1 = {b1:6.3f}   MSE = {np.mean((y_miles - (b0 + b1 * x))**2):7.2f}")

Ahora lo **vemos**: arriba, la recta en distintos pasos sobre los datos; abajo, el camino sobre el tazón y la pérdida que baja.

In [ ]:
fg.descenso_gradiente_paso_a_paso(x, y_miles, eta=0.1, mostrar=(0, 1, 3, 10, 30, 60));

### ¿Llegamos al mismo sitio que la fórmula cerrada?

In [ ]:
from sklearn.linear_model import LinearRegression

cerrada = LinearRegression().fit(x.reshape(-1, 1), y_miles)
print(f"fórmula cerrada (LinearRegression): b0 = {cerrada.intercept_:.3f}   b1 = {cerrada.coef_[0]:.3f}")
print(f"descenso de gradiente, 100 pasos:   b0 = {b0:.3f}   b1 = {b1:.3f}")

### 📘 Para entender — el descenso de gradiente, en serio

**La imagen.** Está en una montaña con niebla y quiere llegar al valle. No ve el valle, pero sí siente con los pies hacia dónde baja el terreno. Da un paso en esa dirección, vuelve a tantear, da otro paso… Eso es el descenso de gradiente.

**La fórmula.** En cada paso, cada parámetro se corrige así:

$$\beta_{\text{nuevo}} = \beta_{\text{actual}} - \eta \cdot \frac{\partial L}{\partial \beta}$$

- $\frac{\partial L}{\partial \beta}$ (la **derivada**, o pendiente) dice cuánto sube la pérdida si $\beta$ aumenta un poquito. Si es positiva, conviene bajar $\beta$; si es negativa, subirlo. Por eso se **resta**.
- $\eta$ (eta), la **tasa de aprendizaje**, es el largo del paso.
- Se repite hasta que los pasos ya casi no cambian nada (la pendiente es ≈ 0: estamos en el fondo).

**¿Si ya existe la fórmula cerrada, para qué sirve?**

| | Fórmula cerrada $(X^\top X)^{-1}X^\top y$ | Descenso de gradiente |
|---|---|---|
| Cómo llega | de un solo golpe, exacto | por pasos, aproximado |
| Cuándo conviene | pocas variables (hasta unos miles) y datos que caben en memoria | millones de filas, miles de variables, datos que llegan por partes |
| Cuándo **no** existe | — | — (es lo único que hay en logística, redes neuronales, t-SNE) |
| Qué hay que elegir | nada | $\eta$ y el número de pasos |

Por eso el descenso de gradiente es **el** motor del ML moderno: la regresión logística (Sesión 5), t-SNE (Sesión 8), el gradient boosting (Sesión 11) y todas las redes neuronales (incluidos los modelos de lenguaje) se entrenan con alguna versión de él.

<details><summary><b>Para quien quiera más:</b> las variantes que va a encontrar</summary>

- **Por lotes (batch)**: cada paso usa todas las filas (lo que hicimos aquí).
- **Estocástico (SGD)**: cada paso usa una sola fila al azar; pasos más baratos y más ruidosos. En scikit-learn: `SGDRegressor`, `SGDClassifier`.
- **Mini-lotes**: cada paso usa, por ejemplo, 32 o 256 filas. Es lo estándar en redes neuronales.
- **Adam, RMSprop**: ajustan el tamaño del paso de cada parámetro por separado. Son los optimizadores de PyTorch y TensorFlow.
</details>

### 🔍 Predice — la perilla del optimizador
El paso de cada iteración es $\eta$ (la **tasa de aprendizaje**). Arriba usamos 0,1 con la edad bien escalada y llegó al fondo. Ahora cambiamos dos cosas: la edad va en **unidades de 30 años** (el tazón se **alarga**) y solo damos **40 pasos**. ¿Qué cree que pasa con cada $\eta$? (no llega · llega · zigzaguea · diverge)

**Su predicción:** η = 0,1: ____ · η = 0,5: ____ · η = 0,95: ____ · η = 1,02: ____

In [ ]:
x30 = ((no_fuman["age"] - no_fuman["age"].mean()) / 30).values   # edad en unidades de 30 años: el tazón se alarga
fg.learning_rate_efecto(x30, y_miles, etas=(0.1, 0.5, 0.95, 1.02), pasos=40);

La tasa de aprendizaje es una perilla **del optimizador**, no del modelo: no cambia qué recta es la mejor, solo cómo se llega a ella. Y el descenso de gradiente es una forma de optimizar entre varias:

![Optimización: el mapa](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S04/img/mapa_optimizacion.png)

### 📘 Los cuatro caminos del mapa, explicados

Todo `.fit()` resuelve el mismo problema: **encontrar los parámetros que hacen más pequeña la pérdida**. Lo que cambia es *cómo*:

| Camino | Cómo funciona | Cuándo se puede usar | En el curso | En la vida real |
|---|---|---|---|---|
| **Fórmula cerrada** | se despeja el mínimo con álgebra, en un paso | cuando la pérdida es un tazón perfecto y existe una fórmula | regresión lineal, Ridge (hoy), PCA (S8) | modelos de precios, pronósticos lineales |
| **Descenso de gradiente** | pasos en contra de la pendiente, de tamaño $\eta$ | cuando la pérdida tiene pendiente (es derivable) pero no hay fórmula | logística (S5), t-SNE (S8), boosting (S11) | redes neuronales, reconocimiento de imágenes, ChatGPT |
| **Búsqueda sin gradiente** | probar candidatos y quedarse con el mejor (todos, uno a uno, alternando o al azar) | cuando no hay pendiente: decisiones con "saltos" (un corte, un grupo) | árboles (S1), K-means (S7), búsqueda de hiperparámetros (S11) | elegir la mejor configuración de un modelo |
| **Con restricciones (PL)** | minimizar un costo **sujeto a reglas** (capacidad, demanda); método Simplex o ramificación y acotamiento | cuando hay que decidir cantidades respetando límites | — (no es tema del curso) | rutas de camiones, cuánto producir en cada planta, horarios de turnos |

> La **programación lineal** (PL) es la prima de la familia: el ML *aprende* parámetros de los datos; la PL *decide* cantidades dados unos costos. En cadena de suministro van juntos: un modelo de ML pronostica la demanda y una PL decide cuánto enviar a cada bodega.

**Convexa o no convexa (la frase de abajo del mapa).** Una pérdida **convexa** tiene un solo fondo, como un tazón: no importa dónde arranque, siempre llega al mismo sitio (lineal, Ridge, logística). Una **no convexa** tiene varios valles: según dónde arranque, termina en uno u otro (K-means, t-SNE, redes). Por eso estos últimos tienen `random_state` y dan resultados distintos con distinta semilla.

## 4. El modelo completo: todas las variables de Insurance

Con varias variables la idea es la misma, en más dimensiones: $\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots$

**One-hot con `drop_first=True`.** En la Sesión 2 creamos una columna por categoría. En un modelo lineal **con intercepto**, una de ellas sobra: si sé que alguien no es del noroeste, ni del sureste, ni del suroeste, ya sé que es del noreste. Esa categoría omitida queda como **referencia** y cada coeficiente se lee *"comparado con la referencia"*.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor

X_ins = pd.get_dummies(ins.drop(columns="charges"), drop_first=True, dtype=int)
y_ins = ins["charges"]
X_tr, X_te, y_tr, y_te = train_test_split(X_ins, y_ins, test_size=0.2, random_state=42)
list(X_ins.columns)

### ✏️ Completa — entrene el modelo y su línea base
La línea base de regresión es `DummyRegressor()`: predice siempre **el promedio** de entrenamiento.

In [ ]:
modelo = ...   # TODO: LinearRegression entrenado con X_tr, y_tr
base = ...     # TODO: DummyRegressor entrenado con X_tr, y_tr
print("Modelo y línea base entrenados.")

### ¿Qué tan bien predice? Tres métricas

![¿Qué tan bien predice un modelo de regresión?](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S04/img/metricas_regresion.png)

### 📘 MAE, RMSE y R²: qué mide cada una y cuál usar

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^n \lvert y_i - \hat{y}_i\rvert \qquad
\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2} \qquad
R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

**Un ejemplo a mano.** Cuatro tiendas; ventas reales $y$ = 10, 20, 30, 40 (cajas) y dos modelos:

| | Tienda 1 | Tienda 2 | Tienda 3 | Tienda 4 | MAE | RMSE |
|---|---|---|---|---|---|---|
| Modelo A (errores parejos) | 12 | 18 | 33 | 37 | **2,5** | 2,55 |
| Modelo B (un error grande) | 10 | 20 | 30 | 30 | **2,5** | **5,0** |

Los dos tienen el **mismo MAE**, pero el B se equivocó feo en una tienda (10 cajas) y el RMSE lo delata. El R² del modelo A: el promedio es 25, la suma de cuadrados alrededor del promedio es 500 y la de los errores es 26 → $R^2 = 1 - 26/500 = 0{,}95$.

| Métrica | Pregunta que responde | Úsela cuando… | Ejemplo |
|---|---|---|---|
| **MAE** | ¿Por cuánto me equivoco, en promedio? | tiene que explicarle el error a alguien no técnico; todos los errores cuestan lo mismo | "el pronóstico de ventas falla por 120 cajas a la semana" |
| **RMSE** | ¿Me equivoco feo de vez en cuando? | los errores grandes son mucho más graves que los pequeños | quedarse sin inventario en diciembre cuesta mucho más que sobrar un poco |
| **R²** | ¿Qué parte de la variación de la *y* explica el modelo? | quiere comparar contra "predecir siempre el promedio" | R² = 0,18 en Saber 11: las condiciones socioeconómicas explican una parte, no todo |

Reglas para no equivocarse:

- Siempre se miden en **prueba** y siempre **contra la línea base** (`DummyRegressor`, que tiene $R^2 = 0$).
- MAE y RMSE están en las **unidades de la *y*** (USD, puntos, cajas); R² no tiene unidades.
- Un R² bajo **no** es un fracaso: el comportamiento humano es difícil de predecir. Un R² de casi 1 es **sospechoso** (sección 7).

### ✏️ Completa — una función que evalúa cualquier modelo de regresión
La va a usar el resto de la sesión. Recibe un nombre, un modelo **ya entrenado** y unos datos; predice y escribe en **una línea** las tres métricas de la ficha (`mean_absolute_error`, `mean_squared_error` —sáquele raíz para el RMSE— y `r2_score`), con un decimal en MAE y RMSE y tres en R².

<details><summary>Pista</summary>

```python
p = m.predict(X)
print(f"{nombre:<22} MAE = {mean_absolute_error(y, p):>8,.1f}   RMSE = {mean_squared_error(y, p) ** 0.5:>8,.1f}   R² = {r2_score(y, p):6.3f}")
```
</details>

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluar(nombre, m, X, y):
    """Imprime en una línea MAE, RMSE y R² del modelo m (ya entrenado) sobre los datos X, y."""
    ...   # TODO: prediga con m y escriba las tres métricas en una línea

evaluar("línea base (promedio)", base, X_te, y_te)
evaluar("regresión lineal", modelo, X_te, y_te)

### Los coeficientes
👀 **Mire:** antes de verlos, ¿cuánto **suma** al cargo anual ser fumador, según el modelo (en USD)? Recuerde la tabla de la sección 1.

In [ ]:
coefs = pd.Series(modelo.coef_, index=X_ins.columns).round(0)
print(f"intercepto: {modelo.intercept_:,.0f}")
coefs.sort_values()

### 📘 Cómo se lee un coeficiente (y cómo no)

- **Numérica:** `age = +248` → "un año más de edad, **con todo lo demás igual**, suma unos 248 USD al cargo anual".
- **Categórica (one-hot):** `smoker_yes = +23 078` → "fumar, **comparado con la categoría de referencia** (no fumar), suma unos 23 000 USD". Sin saber cuál es la referencia, el coeficiente no se puede leer.
- **Intercepto:** la predicción cuando todo vale 0 (edad 0, IMC 0…). Muchas veces no tiene sentido real; es solo el punto de partida de la suma.

Tres errores frecuentes:

1. **Comparar coeficientes de variables en unidades distintas.** El coeficiente de la edad (por año) y el del IMC (por punto) no se comparan directamente. Para comparar "qué pesa más", primero se escalan las variables.
2. **Leerlo como causa.** "Tener más hijos sube el cargo" puede ser cierto en el modelo y deberse a otra cosa que no está en los datos.
3. **Creerle cuando hay variables casi repetidas.** Si dos variables dicen casi lo mismo (en Saber 11, las columnas de vacío de las preguntas de la familia: sección 6), el modelo puede repartir el peso entre ellas de forma rara, hasta con signos opuestos. Se mide con el **VIF** (sección 4b) y la regularización lo estabiliza (sección 6).

### Lo que aprendió, dibujado sobre los datos

In [ ]:
fg.recta_por_grupo(ins.assign(smoker_yes=(ins["smoker"] == "yes").astype(int)), "age", "charges", "smoker",
                   modelo, list(X_ins.columns), etiquetas={"no": "no fuma", "yes": "fuma"},
                   xlabel="edad (años)", ylabel="cargo médico anual (USD)", titulo_grupo="¿fuma?");

### Una fila entra, una predicción sale
Un modelo lineal se explica **sumando**: intercepto + coeficiente × valor de cada variable.

In [ ]:
i = 0
fg.contribuciones_una_fila(modelo, X_te.iloc[i], X_ins.columns, real=y_te.iloc[i], unidad=" USD");

## 4b. ¿Se cumplen los supuestos? Lo que dicen los residuos

Un R² de 0,81 no dice si el modelo está **bien hecho**. La regresión lineal hace cinco promesas sobre sus errores (los **residuos**, $y - \hat{y}$). Cuando no se cumplen pasan dos cosas: el modelo se equivoca **siempre hacia el mismo lado** en ciertas zonas, y los p-valores e intervalos que reportaría un programa de estadística **dejan de ser confiables**.

![Los supuestos de la regresión lineal](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S04/img/supuestos_regresion_ficha.png)

Dos reglas para leerlos:
- Se revisan **después** del `.fit()`, con los residuos del conjunto de **prueba**.
- **Son sobre los errores, no sobre la *y* ni sobre las X.** Que `charges` sea asimétrica no rompe nada por sí sola; lo que importa es cómo quedan los residuos.

### 🔍 Predice — ¿qué supuestos fallan?
Ya tiene dos pistas: en la figura de las rectas los fumadores forman **dos nubes** y la recta pasa entre ellas, y el RMSE (≈ 6 000) es mucho mayor que el MAE (≈ 4 200). Antes de correr, marque cuáles cree que fallan:

| Supuesto | ¿Falla? (sí / no) |
|---|---|
| 1 Linealidad | |
| 2 Independencia | |
| 3 Homocedasticidad | |
| 4 Normalidad de los errores | |
| 5 Multicolinealidad | |

In [ ]:
residuos = y_te - modelo.predict(X_te)
fg.diagnostico_supuestos(y_te, modelo.predict(X_te));

In [ ]:
fg.vif(X_tr).round(2)     # supuesto 5: ¿alguna variable es (casi) copia de las otras?

### ¿De dónde sale el patrón? Residuos por grupo
Si al modelo le falta algo, los residuos lo delatan. Agrupemos por las dos variables de la pista: fumar y tener obesidad (IMC ≥ 30).

In [ ]:
(X_te.assign(residuo=residuos, obeso=X_te["bmi"] >= 30)
     .groupby(["smoker_yes", "obeso"])["residuo"].agg(["mean", "count"]).round(0))

### ✏️ Completa — dele al modelo la columna que le falta
Cree `fuma_obeso`: vale 1 si la persona fuma **y** tiene IMC ≥ 30, y 0 si no. Agréguela a `X_tr` y a `X_te`, vuelva a entrenar y repita el diagnóstico.

<details><summary>Pista</summary>

```python
X_tr2 = X_tr.assign(fuma_obeso=X_tr["smoker_yes"] * (X_tr["bmi"] >= 30).astype(int))
```
</details>

In [ ]:
X_tr2 = ...   # TODO: X_tr con la columna fuma_obeso (fuma Y bmi >= 30)
X_te2 = ...   # TODO: lo mismo para X_te
modelo2 = LinearRegression().fit(X_tr2, y_tr)
evaluar("regresión lineal", modelo, X_te, y_te)
evaluar("+ fuma_obeso", modelo2, X_te2, y_te)
fg.diagnostico_supuestos(y_te, modelo2.predict(X_te2));

In [ ]:
pd.DataFrame({"sin fuma_obeso": pd.Series(modelo.coef_, index=X_tr.columns),
              "con fuma_obeso": pd.Series(modelo2.coef_, index=X_tr2.columns)}).reindex(X_tr2.columns).round(0)

### 📘 Para entender — los supuestos: de dónde salen y cuándo importan

**De dónde salen.** Con los supuestos 1, 2, 3 y 5, el teorema de **Gauss–Markov** garantiza que mínimos cuadrados da el **mejor estimador lineal insesgado**: ningún otro método lineal sin sesgo tiene coeficientes con menos varianza. El 4 (normalidad) se agrega para que los p-valores y los intervalos sean **exactos** con pocos datos; con muchos datos, el teorema del límite central hace casi todo el trabajo. Ojo: solo la multicolinealidad **perfecta** rompe el supuesto 5 (una columna que es combinación exacta de otras, como olvidar `drop_first`); la alta es un problema práctico, no una violación.

**Predecir no es lo mismo que interpretar.**

| Supuesto | Si falla, ¿se daña la predicción? | ¿Y la interpretación (β, p-valores, intervalos)? |
|---|---|---|
| 1 Linealidad | **Sí**: errores sistemáticos en ciertas zonas | **Sí**: los β describen una recta que no es |
| 2 Independencia | la predicción poco, pero la **validación** promete de más | **Sí**: errores estándar demasiado pequeños |
| 3 Homocedasticidad | poco | **Sí**: p-valores e intervalos equivocados |
| 4 Normalidad | poco | con pocos datos, sí |
| 5 Multicolinealidad | poco (dentro del rango de los datos) | **Sí**: coeficientes inestables, signos que cambian |

En machine learning casi siempre se juzga un modelo por cómo **predice en prueba**, y por eso los supuestos se nombran menos que en un curso de estadística. Pero un patrón en los residuos sigue siendo la mejor pista de **qué le falta al modelo**, como acaba de ver con `fuma_obeso`.

**Qué hacer cuando falla cada uno**

- **Linealidad:** transformar una x (`np.log`, $x^2$), agregar una interacción, o usar un modelo no lineal (árboles, Sesión 6).
- **Independencia:** validar respetando el tiempo o el grupo (`TimeSeriesSplit`, `GroupKFold`, Sesión 10).
- **Homocedasticidad:** modelar `log(y)` si el error crece en proporción a la *y*, o usar errores estándar robustos (HC3).
- **Normalidad:** con muchos datos casi no importa; si necesita intervalos, use *bootstrap*.
- **Multicolinealidad:** quitar o combinar variables, Ridge, o PCA (Sesión 8).

**Si quiere las pruebas formales** (`statsmodels` ya viene en Colab):

```python
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
ols = sm.OLS(y_tr, sm.add_constant(X_tr)).fit()
print(ols.summary())                                    # coeficientes con p-valores e intervalos
print(het_breuschpagan(ols.resid, ols.model.exog)[1])   # p-valor de Breusch–Pagan: < 0,05 → hay embudo
```

Con muchos datos, estas pruebas rechazan hasta por diferencias mínimas: **el gráfico dice cuánto falla; la prueba solo dice si falla.**

### ⚡ Mini-reto (opcional)
Sin la columna nueva, pruebe modelar el **logaritmo** del cargo: entrene con `np.log(y_tr)` y deshaga con `np.exp(...)` al predecir. ¿Mejora el R² medido en dólares? ¿Por qué cree que sí o que no? (Los árboles de la Sesión 6 encuentran cortes como "fuma e IMC ≥ 30" por su cuenta; la regresión lineal necesita que usted le dé la columna.)

## 5. Ahora con datos colombianos: Saber 11, Valle del Cauca, 2022

78 720 estudiantes que presentaron la prueba en el segundo semestre de 2022 (calendario A) en colegios del Valle. La pregunta: **¿cuánto del puntaje global se explica por las condiciones del estudiante, su familia y su colegio?**

In [ ]:
s11 = pd.read_csv("saber11_valle_2022.csv.gz")
print(s11.shape)
s11.head(3)

### ✏️ Completa — aplique la Sesión 2
Mire rápido el dataset: ¿qué columnas tienen vacíos y cuántos? (una línea). Fíjese en `fami_estratovivienda` y en las de educación de los padres.

In [ ]:
...   # TODO: % de vacíos por columna, de mayor a menor (Sesión 2)

In [ ]:
print(s11["fami_estratovivienda"].value_counts(dropna=False))
print("\npunt_global:", s11["punt_global"].describe().round(1).to_dict())

Para X usamos variables del colegio y de la familia, y la edad. Las categóricas van a *one-hot* con `dummy_na=True` (el vacío es una categoría más, como en la Sesión 2) y `drop_first=True` (sección 4): en cada variable la **primera categoría en orden alfabético** queda como **referencia**; en el estrato es *Estrato 1*. Dos detalles más: las columnas que valen lo mismo en todas las filas (el "vacío" de variables que no tienen vacíos) no aportan nada y se quitan, y la edad faltante (32 estudiantes) se llena con la mediana **de entrenamiento** (Sesión 3). La *y* es `punt_global` (0 a 500).

In [ ]:
categoricas = ["cole_area_ubicacion", "cole_bilingue", "cole_caracter", "cole_jornada", "cole_naturaleza", "estu_genero",
               "fami_educacionmadre", "fami_educacionpadre", "fami_estratovivienda", "fami_tieneautomovil",
               "fami_tienecomputador", "fami_tieneinternet", "fami_tienelavadora", "fami_personashogar", "fami_cuartoshogar"]
X_s = pd.get_dummies(s11[categoricas + ["edad"]], columns=categoricas, dummy_na=True, drop_first=True, dtype=int)
X_s = X_s.loc[:, X_s.nunique() > 1]                    # fuera las columnas constantes
y_s = s11["punt_global"]
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(X_s, y_s, test_size=0.2, random_state=42)
mediana_edad = Xs_tr["edad"].median()                  # la mediana sale SOLO de entrenamiento
Xs_tr, Xs_te = Xs_tr.fillna({"edad": mediana_edad}), Xs_te.fillna({"edad": mediana_edad})
print("X:", X_s.shape)

👀 **Mire:** con 65 columnas socioeconómicas y del colegio, ¿qué R² cree que logra la regresión lineal? ¿0,2? ¿0,5? ¿0,8?

In [ ]:
lin_s = LinearRegression().fit(Xs_tr, ys_tr)
base_s = DummyRegressor().fit(Xs_tr, ys_tr)
evaluar("línea base (promedio)", base_s, Xs_te, ys_te)
evaluar("regresión lineal", lin_s, Xs_te, ys_te)

### 🔍 Predice — ¿le cree al coeficiente?
Comparado con un estudiante de **estrato 1** (la categoría de referencia), y con todo lo demás igual, ¿cuántos puntos **más o menos** cree que predice el modelo para uno de **estrato 6**?

**Su predicción:** ____

In [ ]:
estrato = pd.Series(lin_s.coef_, index=X_s.columns).filter(like="estrato").round(1)
print("Puntos comparado con Estrato 1 (la referencia):")
print(estrato.sort_values())
print("\nPromedio real de punt_global por estrato:")
print(s11.groupby("fami_estratovivienda")["punt_global"].agg(["mean", "count"]).round(1))

**¡Estrato 6 unos 30 puntos por debajo de estrato 1!** Y no es un error del modelo: el promedio crudo también lo dice (213 contra 243). Antes de publicar *"en el Valle, los estudiantes de estrato 6 sacan menos"*, la pregunta de la Sesión 2: **¿cambió el mundo o cambió el dato?**

In [ ]:
perfil = s11[s11["fami_estratovivienda"].isin(["Estrato 1", "Estrato 3", "Estrato 6"])].groupby("fami_estratovivienda").agg(
    estudiantes=("punt_global", "size"),
    colegio_oficial=("cole_naturaleza", lambda v: (v == "OFICIAL").mean()),
    tiene_internet=("fami_tieneinternet", lambda v: (v == "Si").mean()),
    tiene_carro=("fami_tieneautomovil", lambda v: (v == "Si").mean()),
).round(2)
perfil

## 6. Regularización: Ridge y Lasso

Con 65 columnas, algunas **cuentan casi lo mismo**, y eso vuelve los coeficientes **inestables** y difíciles de leer: es la **multicolinealidad**, el supuesto 5 de la ficha. Antes de regularizar, midámosla con el VIF.

👀 **Mire:** ¿cuáles cree que salen más redundantes? ¿Educación de la madre y del padre? ¿Computador e internet?

In [ ]:
vif_s = fg.vif(Xs_tr)
print(f"{(vif_s > 10).sum()} columnas con VIF > 10 y {(vif_s > 5).sum()} con VIF > 5, de {len(vif_s)}")
vif_s.head(10).round(1)

La regularización agrega a la pérdida un **castigo por coeficientes grandes**:

$$\text{Ridge: } \sum (y_i - \hat{y}_i)^2 + \alpha \sum_j \beta_j^2 \qquad\qquad \text{Lasso: } \frac{1}{2n}\sum (y_i - \hat{y}_i)^2 + \alpha \sum_j |\beta_j|$$

(Así las define scikit-learn: Lasso divide la suma de cuadrados por $2n$ y Ridge no. **Por eso el $\alpha$ de Lasso y el de Ridge no están en la misma escala**: con nuestros 62 976 estudiantes de entrenamiento, un $\alpha$ de Lasso equivale a uno unas 126 000 veces más grande en Ridge. La figura de abajo los pone en el mismo eje.)

- $\alpha$ es la **fuerza** del castigo: $\alpha = 0$ es la regresión de siempre; $\alpha$ muy grande aplasta todos los coeficientes a 0.
- **Ridge** encoge todos los coeficientes; **Lasso** apaga los menos útiles (los vuelve exactamente 0): **elige variables**.
- Ojo: el castigo depende del tamaño del coeficiente, y el tamaño depende de la escala de la variable. Por eso **se escala antes** (Sesión 2). Aquí $\alpha$ es una perilla **del modelo**, no del optimizador.

<details><summary><b>Para quien quiera más:</b> ¿por qué Lasso apaga y Ridge no?</summary>

El castigo $|\beta|$ tiene una "punta" en 0: cerca de cero, sigue empujando con la misma fuerza hasta dejar el coeficiente exactamente en 0. El castigo $\beta^2$ se vuelve muy suave cerca de 0 (su pendiente es $2\beta$, que se anula), así que encoge pero casi nunca llega a cero.
</details>

![Ridge y Lasso: la regresión que le cobra a los coeficientes grandes](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S04/img/ridge_lasso_ficha.png)

### 🔍 Predice
Si subimos $\alpha$ poco a poco, ¿qué pasará con el R² de prueba? ¿Baja de inmediato, se mantiene un rato, sube?

**Su predicción:** ____

In [ ]:
fg.regularizacion_alpha(Xs_tr, ys_tr, Xs_te, ys_te, alphas=np.logspace(-2, 1.2, 12));

**Lo que pasa:** el R² **se mantiene** un buen tramo mientras los coeficientes se encogen o se apagan, y solo cae cuando el castigo ya es muy fuerte. Muchas variables sobraban.

### ✏️ Completa — Lasso como selector de variables
Entrene un `Lasso` con `alpha=2` (y `max_iter=10000`, para que el optimizador alcance a llegar) sobre las variables **escaladas** de entrenamiento (el escalador ya se ajustó solo con entrenamiento, Sesión 3). Luego arme `vivos`: la Serie de coeficientes con los nombres de las columnas, quedándose solo con los distintos de cero.

<details><summary>Pista</summary>

```python
lasso = Lasso(alpha=2, max_iter=10000).fit(Zs_tr, ys_tr)
vivos = pd.Series(lasso.coef_, index=X_s.columns)
vivos = vivos[vivos != 0]
```
</details>

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

escalador = StandardScaler().fit(Xs_tr)
Zs_tr, Zs_te = escalador.transform(Xs_tr), escalador.transform(Xs_te)
lasso = ...   # TODO: Lasso con alpha=2 entrenado sobre las variables escaladas de entrenamiento
vivos = ...   # TODO: coeficientes de lasso con su nombre de columna, solo los distintos de 0
print(f"Lasso conserva {len(vivos)} de {X_s.shape[1]} variables · R² prueba = {lasso.score(Zs_te, ys_te):.3f} (lineal: {lin_s.score(Xs_te, ys_te):.3f})")
vivos.reindex(vivos.abs().sort_values(ascending=False).index).head(8).round(2)

### 📘 ¿Ridge, Lasso o ninguno? Una guía para decidir

| Situación | Elija | Por qué |
|---|---|---|
| Pocas variables, poco relacionadas entre sí | `LinearRegression` | no hay nada que castigar; los coeficientes se leen tal cual |
| Muchas variables **parecidas entre sí** y quiere usarlas todas | **Ridge** | reparte el peso entre las parecidas y estabiliza los coeficientes |
| Muchas variables y sospecha que **pocas importan** | **Lasso** | apaga las que sobran: el modelo queda más corto y más fácil de explicar |
| Las dos cosas a la vez | **Elastic Net** (`ElasticNet`) | mezcla los dos castigos |
| Más variables que filas | Ridge, Lasso o Elastic Net | sin castigo, la regresión ni siquiera tiene una única solución |

Ejemplos de uso:

- **Pronóstico de demanda** con 200 variables de clima, precios, festivos y promociones de la competencia: Lasso para quedarse con las 15 que mueven las ventas.
- **Precio de vivienda** con muchas medidas del inmueble que se repiten (m² construidos, m² totales, número de habitaciones): Ridge para que los coeficientes no salten.
- **Encuestas** como el Saber 11, con muchas preguntas socioeconómicas parecidas: Lasso para ver cuáles sobreviven.

Y siempre:

1. **Escale antes** (`StandardScaler`, ajustado solo con entrenamiento): el castigo depende del tamaño del coeficiente, y el tamaño depende de las unidades.
2. **Elija $\alpha$ con validación**, no a ojo. scikit-learn trae `RidgeCV` y `LassoCV`, que prueban varios $\alpha$ solos (la validación cruzada es el tema de la Sesión 10).

## 7. 💥 Cinco columnas sin usar

El R² de 0,18 deja un sabor amargo. En el dataset hay cinco columnas más que no hemos usado: los **puntajes por área** (matemáticas, lectura crítica, sociales, ciencias naturales, inglés). Agreguémoslas a X a ver cuánto mejora.

### 🔍 Predice
¿Qué R² cree que logra el modelo con los puntajes por área?

**Su predicción:** R² ≈ ____

In [ ]:
areas = ["punt_matematicas", "punt_lectura_critica", "punt_sociales_ciudadanas", "punt_c_naturales", "punt_ingles"]
X_mas = pd.concat([X_s, s11[areas]], axis=1)
Xm_tr, Xm_te, _, _ = train_test_split(X_mas, y_s, test_size=0.2, random_state=42)
medianas = Xm_tr.median()                              # otra vez: medianas solo de entrenamiento
Xm_tr, Xm_te = Xm_tr.fillna(medianas), Xm_te.fillna(medianas)
lin_mas = LinearRegression().fit(Xm_tr, ys_tr)
evaluar("lineal + puntajes", lin_mas, Xm_te, ys_te)

In [ ]:
fg.pred_vs_real({"socioeconómicas (honesto)": (ys_te, lin_s.predict(Xs_te)),
                 "+ puntajes por área": (ys_te, lin_mas.predict(Xm_te))}, unidad=" pts");

**R² = 1,000 y un error de 0,2 puntos.** El modelo perfecto. ¿Qué aprendió? Miremos los coeficientes de los puntajes:

In [ ]:
pd.Series(lin_mas.coef_, index=X_mas.columns)[areas].round(3)

## 8. Cierre — qué cambió hoy en el pipeline

| Etapa | Hoy (Sesión 4) |
|---|---|
| Datos | dos datasets: uno de referencia (Insurance) y uno colombiano (Saber 11); duplicados y vacíos con lo de la Sesión 2 |
| Preprocesamiento | `drop_first` en modelos con intercepto · escalar antes de Ridge/Lasso y del descenso de gradiente |
| Modelo | **regresión lineal**: pérdida cuadrática · fórmula cerrada vs **descenso de gradiente** · **tasa de aprendizaje** · Ridge y Lasso |
| Evaluación | **MAE, RMSE, R²** contra `DummyRegressor` · un R² ≈ 1 es sospechoso · **supuestos con los residuos** (linealidad, independencia, homocedasticidad, normalidad) y **VIF** |
| Interpretación | coeficientes "manteniendo lo demás fijo" · una predicción = suma de contribuciones · un patrón en los residuos = algo que le falta al modelo (**interacción** fuma × obesidad) · un coeficiente raro es primero una pregunta sobre el dato |

**Dentro del `.fit()` — lo que ya puede comparar:**

| | Árbol (S1–S3) | Regresión lineal (S4) |
|---|---|---|
| Qué minimiza | Gini de cada corte | suma de residuales² (+ castigo en Ridge/Lasso) |
| Cómo lo busca | búsqueda voraz (exhaustiva en cada corte) | fórmula cerrada o descenso de gradiente |
| Qué perilla controla la búsqueda | ninguna | tasa de aprendizaje η (si hay gradiente) |
| Perilla del modelo | `max_depth` | `alpha` |

**Para la Sesión 5 — clasificación:** volvemos a una *y* de sí/no, con un dataset nuevo (una campaña de mercadeo bancario). ¿Qué pasa si a una pregunta de sí/no le ajustamos una recta? Spoiler: se sale de 0 y 1, y hay que doblarla.

### 📘 La regresión lineal en una página (para repasar)

| Concepto | Qué es | Fórmula / código | Para qué lo uso |
|---|---|---|---|
| Regresión lineal | predice un número como suma ponderada | $\hat y = \beta_0 + \sum \beta_j x_j$ · `LinearRegression()` | predecir y explicar una *y* numérica |
| Residual | lo que se equivoca en una fila | $y_i - \hat y_i$ | ver dónde y cuánto falla |
| Pérdida (MSE) | el promedio de residuales² | $\frac1n\sum (y_i-\hat y_i)^2$ | es lo que el `.fit()` minimiza |
| Fórmula cerrada | el mínimo, despejado con álgebra | $(X^\top X)^{-1}X^\top y$ | regresión con pocas variables |
| Descenso de gradiente | pasos en contra de la pendiente | $\beta \leftarrow \beta - \eta\,\nabla L$ | modelos sin fórmula; datos enormes |
| Tasa de aprendizaje $\eta$ | el tamaño del paso | perilla del optimizador | muy chica: no llega · muy grande: diverge |
| `drop_first` | quitar una categoría por variable | `get_dummies(..., drop_first=True)` | modelos con intercepto; la quitada es la referencia |
| MAE / RMSE / R² | qué tan bien predice | `mean_absolute_error`, `mean_squared_error`, `r2_score` | MAE para explicar, RMSE si los errores grandes duelen, R² contra el promedio |
| `DummyRegressor` | la línea base: predice el promedio | `DummyRegressor()` | saber si el modelo aprendió algo |
| Ridge / Lasso | regresión + castigo a coeficientes grandes | $+\alpha\sum\beta_j^2$ · $+\alpha\sum\lvert\beta_j\rvert$ | estabilizar (Ridge) o elegir variables (Lasso) |
| Columna del futuro | una *x* que se conoce después de la *y* o está hecha de ella | — | sospechar siempre de un R² ≈ 1 |